# VehiGuard - train the Indian plate detector (task 2.4)

Run this notebook on **Google Colab with a free GPU**:
*Runtime -> Change runtime type -> T4 GPU*, then *Runtime -> Run all*.

It downloads your labelled dataset from Roboflow (task 2.3, see `docs/plate_dataset_guide.md`),
fine-tunes YOLOv8 starting from the Nepal baseline (D07), reports mAP, precision and recall,
and downloads `best.pt` for you.

In [ ]:
# Same Ultralytics version as the laptop, so the model loads there without problems
!pip install -q ultralytics==8.4.174 roboflow huggingface_hub
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

## 1. Get the dataset - run ONE of the two cells below

**Option A (simplest): upload the zip made on the laptop.** On the laptop run
`python scripts/make_yolo_dataset.py data/train_plates data/datasets/datacluster_eval/labels data/datasets/plates_yolo`,
then run the next cell and choose `data/datasets/plates_yolo.zip` when asked.

**Option B: download from Roboflow** (if you labelled or added images there): skip the
next cell and run the one after it.

In [ ]:
# Option A: upload plates_yolo.zip from the laptop
import zipfile
from types import SimpleNamespace

from google.colab import files

uploaded = files.upload()  # choose plates_yolo.zip
name = next(iter(uploaded))
zipfile.ZipFile(name).extractall("/content")
dataset = SimpleNamespace(location="/content/" + name.removesuffix(".zip"))
# make the dataset path absolute for training on Colab
yaml_path = f"{dataset.location}/data.yaml"
with open(yaml_path) as f:
    text = f.read().replace("path: plates_yolo", f"path: {dataset.location}")
with open(yaml_path, "w") as f:
    f.write(text)
print("Dataset in", dataset.location)
!cat {yaml_path}
!echo "train images:" && ls {dataset.location}/train/images | wc -l
!echo "valid images:" && ls {dataset.location}/valid/images | wc -l

In [ ]:
# Option B: download from Roboflow (version -> Download Dataset -> YOLOv8 -> show code)
from getpass import getpass

from roboflow import Roboflow

WORKSPACE = "your-workspace"   # <- change
PROJECT = "indian-plates"      # <- change
VERSION = 1                    # <- change

rf = Roboflow(api_key=getpass("Roboflow API key: "))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
print("Dataset in", dataset.location)
!cat {dataset.location}/data.yaml

## 2. Fine-tune

Starts from the Nepal plate model (it already knows what plates look like), so it needs
fewer images and epochs. Training stops early when it stops improving.

In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

start = hf_hub_download("krishnamishra8848/Nepal-Vehicle-License-Plate-Detection", "last.pt")
model = YOLO(start)
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60, patience=15, imgsz=640, batch=16,
    project="runs", name="plate", exist_ok=True,
)

## 3. Results on the validation split

These photos were held back from training. The real test for your report is
`scripts/evaluate.py` on your own photos (step 5), which the model has never seen.

In [ ]:
best = YOLO("runs/plate/weights/best.pt")
metrics = best.val(data=f"{dataset.location}/data.yaml", split="val")
print(f"mAP50     {metrics.box.map50:.3f}")
print(f"mAP50-95  {metrics.box.map:.3f}")
print(f"Precision {metrics.box.mp:.3f}")
print(f"Recall    {metrics.box.mr:.3f}")

In [ ]:
from IPython.display import Image, display

for name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    try:
        display(Image(filename=f"runs/plate/{name}", width=900))
    except FileNotFoundError:
        pass

## 4. Download the model

In [ ]:
from google.colab import files

files.download("runs/plate/weights/best.pt")

## 5. Use it on the laptop

1. Rename the downloaded file to `plate_detector.pt` and put it in the `models/` folder of the project.
2. Compare it with the Nepal baseline on your own test photos (data/test_plates, never used for training):
   ```
   python scripts/evaluate.py --name baseline
   python scripts/evaluate.py --weights models/plate_detector.pt --name finetuned
   ```
3. If it is better, set `plate_model: models/plate_detector.pt` in `config/settings.yaml`.
4. Record the model in `docs/datasets.md` (name, date, dataset version, metrics).